# Deploy One Model to Three AzureML Endpoint Types

This notebook demonstrates how to deploy the same machine learning model to three different AzureML endpoint types using Azure ML SDK v2:

- **🅰️ Managed Online Endpoint** - Real-time inference with managed infrastructure
- **🅱️ Serverless Endpoint** - Pay-per-use serverless inference
- **🅾️ Batch Endpoint** - Large-scale batch scoring

The notebook also shows how to consume each endpoint from Databricks.

## Prerequisites

- Azure subscription with AzureML workspace
- Appropriate permissions to create endpoints
- Databricks workspace with runtime 13.0+
- Azure authentication configured (Service Principal or Managed Identity)

---

## SECTION 1 — Install Dependencies

Install the required Azure ML SDK v2 packages and dependencies.

In [ ]:
%pip install azure-ai-ml azure-identity requests scikit-learn

---

## SECTION 2 — Configure AzureML Client

Set up authentication and create an MLClient instance to interact with Azure ML workspace.

**Important**: Replace the placeholders below with your actual Azure values:
- `<SUBSCRIPTION_ID>` - Your Azure subscription ID
- `<RESOURCE_GROUP>` - Your Azure resource group name
- `<AML_WORKSPACE>` - Your Azure ML workspace name

For production, retrieve these from environment variables or Azure Key Vault.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import os

# Configuration - Replace with your values
subscription_id = "<SUBSCRIPTION_ID>"
resource_group = "<RESOURCE_GROUP>"
workspace_name = "<AML_WORKSPACE>"

# Authenticate using DefaultAzureCredential
# This will use Managed Identity, Service Principal, or other available credentials
credential = DefaultAzureCredential()

# Create MLClient instance
ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name
)

print("✓ MLClient created successfully")
print(f"  Workspace: {workspace_name}")
print(f"  Resource Group: {resource_group}")
print(f"  Subscription: {subscription_id}")

---

## SECTION 3 — Load or Create a Simple Model

Create a simple scikit-learn Logistic Regression model using the Iris dataset.
This model will be deployed to all three endpoint types.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
import pickle
import os

# Load Iris dataset
X, y = load_iris(return_X_y=True)

# Train a simple Logistic Regression model
clf = LogisticRegression(max_iter=200, random_state=42)
clf.fit(X, y)

# Create model directory if it doesn't exist
os.makedirs("model", exist_ok=True)

# Save the model
model_path = "model/model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(clf, f)

print("✓ Model trained and saved successfully")
print(f"  Model path: {model_path}")
print(f"  Training accuracy: {clf.score(X, y):.4f}")
print(f"  Model classes: {clf.classes_}")

Create a scoring script for the model:

In [ ]:
%%writefile model/score.py
import json
import pickle
import numpy as np
import os

def init():
    global model
    # Get the path to the registered model file
    model_path = os.path.join(os.getenv('AZUREML_MODEL_DIR', ''), 'model.pkl')
    # Load the model
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    print("Model loaded successfully")

def run(raw_data):
    try:
        # Parse input data
        data = json.loads(raw_data)
        
        # Handle different input formats
        if 'input_data' in data:
            input_data = data['input_data']
            if isinstance(input_data, list):
                np_data = np.array(input_data)
            elif isinstance(input_data, dict) and 'data' in input_data:
                np_data = np.array(input_data['data'])
            else:
                np_data = np.array(input_data)
        else:
            np_data = np.array(data)
        
        # Make prediction
        predictions = model.predict(np_data)
        probabilities = model.predict_proba(np_data)
        
        # Return predictions
        return json.dumps({
            'predictions': predictions.tolist(),
            'probabilities': probabilities.tolist()
        })
    except Exception as e:
        return json.dumps({'error': str(e)})

Register the model in Azure ML:

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

# Register the model
model = Model(
    path="model",
    type=AssetTypes.CUSTOM_MODEL,
    name="iris-classifier",
    description="Iris classifier trained with scikit-learn LogisticRegression",
)

registered_model = ml_client.models.create_or_update(model)

print("✓ Model registered successfully")
print(f"  Name: {registered_model.name}")
print(f"  Version: {registered_model.version}")
print(f"  ID: {registered_model.id}")

---

## 🅰️ SECTION 4 — Deploy Model to Managed Online Endpoint (A)

Managed Online Endpoints provide real-time inference with fully managed infrastructure.
Azure handles scaling, monitoring, and infrastructure management.

**Steps:**
1. Create a Managed Online Endpoint
2. Create a Managed Online Deployment
3. Allocate 100% traffic to the deployment
4. Retrieve scoring URI and authentication key

### Step 4.1: Create Managed Online Endpoint

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint
import time

# Define endpoint name - Replace with your desired name
managed_endpoint_name = "<MANAGED_ENDPOINT_NAME>"  # e.g., "iris-managed-endpoint"

# Create Managed Online Endpoint
managed_endpoint = ManagedOnlineEndpoint(
    name=managed_endpoint_name,
    description="Managed online endpoint for Iris classifier",
    auth_mode="key",  # Use key-based authentication
    tags={"model": "iris-classifier", "type": "managed-online"}
)

# Create the endpoint
print(f"Creating Managed Online Endpoint: {managed_endpoint_name}...")
managed_endpoint_result = ml_client.online_endpoints.begin_create_or_update(managed_endpoint).result()

print("✓ Managed Online Endpoint created successfully")
print(f"  Name: {managed_endpoint_result.name}")
print(f"  Scoring URI: {managed_endpoint_result.scoring_uri}")
print(f"  Auth Mode: {managed_endpoint_result.auth_mode}")

### Step 4.2: Create Managed Online Deployment

In [ ]:
from azure.ai.ml.entities import ManagedOnlineDeployment, Environment, CodeConfiguration

# Create deployment
managed_deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=managed_endpoint_name,
    model=registered_model.id,
    code_configuration=CodeConfiguration(
        code="model",
        scoring_script="score.py"
    ),
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu:1",
    instance_type="Standard_DS2_v2",
    instance_count=1,
)

print(f"Creating deployment 'blue' for endpoint {managed_endpoint_name}...")
print("This may take 10-15 minutes...")
managed_deployment_result = ml_client.online_deployments.begin_create_or_update(managed_deployment).result()

print("✓ Managed Online Deployment created successfully")
print(f"  Deployment name: {managed_deployment_result.name}")
print(f"  Instance type: {managed_deployment_result.instance_type}")
print(f"  Instance count: {managed_deployment_result.instance_count}")

### Step 4.3: Allocate 100% Traffic to Deployment

In [ ]:
# Set traffic to 100% for the blue deployment
managed_endpoint_result.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(managed_endpoint_result).result()

print("✓ Traffic allocated to deployment")
print(f"  blue: 100%")

### Step 4.4: Retrieve Scoring URI and Authentication Key

In [ ]:
# Get the endpoint details
managed_endpoint_details = ml_client.online_endpoints.get(managed_endpoint_name)
managed_scoring_uri = managed_endpoint_details.scoring_uri

# Get the primary key
managed_keys = ml_client.online_endpoints.get_keys(managed_endpoint_name)
managed_primary_key = managed_keys.primary_key

print("✓ Managed Online Endpoint credentials retrieved")
print(f"  Scoring URI: {managed_scoring_uri}")
print(f"  Primary Key: <PRIMARY_KEY>  # Hidden for security")
print("\n⚠️  IMPORTANT: Store these values securely")
print("  - Do not print managed_primary_key in production")
print("  - Use Azure Key Vault or Databricks Secrets for storage")
print("  - You'll need these credentials for consumption in Section 7")

---

## 🅱️ SECTION 5 — Deploy Model to Serverless Endpoint (B)

Serverless Endpoints provide pay-per-use inference without managing infrastructure.
Ideal for unpredictable workloads with automatic scaling to zero.

**Note**: Serverless endpoints are available for specific model types and regions.
For custom models, Managed Online Endpoints are typically used.

**Steps:**
1. Create a Serverless Endpoint
2. Create a Serverless Deployment
3. Retrieve scoring URI and authentication key

### Step 5.1: Create Serverless Endpoint

**Important Note**: As of SDK v2, Azure ML uses `ServerlessEndpoint` for serverless deployments.
Serverless endpoints are primarily for models from the Azure ML model catalog.

For custom models like our Iris classifier, we typically use Managed Online Endpoints.
This section demonstrates the API structure for when serverless is available.

In [ ]:
# Define serverless endpoint name - Replace with your desired name
serverless_endpoint_name = "<SERVERLESS_ENDPOINT_NAME>"  # e.g., "iris-serverless-endpoint"

# Note: Serverless endpoints are typically used with models from Azure ML model catalog
# For demonstration, this shows the structure. Actual deployment may require catalog models.

print("\n⚠️  Serverless Endpoint Note:")
print("Serverless endpoints are primarily for models from Azure ML model catalog.")
print("For custom models, Managed Online Endpoints are recommended.")
print("\nIf you have a catalog model ID, you can create a serverless endpoint like this:")

# Example structure (commented out as it requires a catalog model)
# from azure.ai.ml.entities import ServerlessEndpoint
# 
# serverless_endpoint = ServerlessEndpoint(
#     name=serverless_endpoint_name,
#     model_id="azureml://registries/azureml/models/your-catalog-model/versions/1",
#     auth_mode="key"
# )
# 
# serverless_endpoint_result = ml_client.serverless_endpoints.begin_create_or_update(serverless_endpoint).result()
# print(f"✓ Serverless Endpoint created: {serverless_endpoint_result.name}")

### Alternative: Create Second Managed Online Endpoint as Serverless Pattern

For custom models, we can create a second managed online endpoint configured for serverless-like behavior
(low minimum instances, aggressive scale-down).

In [ ]:
# Create a second managed endpoint with serverless-like configuration
serverless_like_endpoint_name = "<SERVERLESS_ENDPOINT_NAME>"  # e.g., "iris-serverless-like"

serverless_like_endpoint = ManagedOnlineEndpoint(
    name=serverless_like_endpoint_name,
    description="Serverless-like endpoint for Iris classifier (Managed with auto-scaling)",
    auth_mode="key",
    tags={"model": "iris-classifier", "type": "serverless-pattern"}
)

print(f"Creating Serverless-like Managed Endpoint: {serverless_like_endpoint_name}...")
serverless_like_endpoint_result = ml_client.online_endpoints.begin_create_or_update(serverless_like_endpoint).result()

print("✓ Serverless-like Endpoint created successfully")
print(f"  Name: {serverless_like_endpoint_result.name}")
print(f"  Scoring URI: {serverless_like_endpoint_result.scoring_uri}")

### Step 5.2: Create Deployment for Serverless-like Endpoint

In [ ]:
# Create deployment with serverless-like configuration
serverless_like_deployment = ManagedOnlineDeployment(
    name="green",
    endpoint_name=serverless_like_endpoint_name,
    model=registered_model.id,
    code_configuration=CodeConfiguration(
        code="model",
        scoring_script="score.py"
    ),
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu:1",
    instance_type="Standard_DS2_v2",
    instance_count=1,
)

print(f"Creating deployment 'green' for serverless-like endpoint...")
print("This may take 10-15 minutes...")
serverless_like_deployment_result = ml_client.online_deployments.begin_create_or_update(serverless_like_deployment).result()

# Set traffic to 100%
serverless_like_endpoint_result.traffic = {"green": 100}
ml_client.online_endpoints.begin_create_or_update(serverless_like_endpoint_result).result()

print("✓ Serverless-like Deployment created successfully")
print(f"  Deployment name: {serverless_like_deployment_result.name}")

### Step 5.3: Retrieve Scoring URI and Key for Serverless-like Endpoint

In [ ]:
# Get endpoint details
serverless_like_endpoint_details = ml_client.online_endpoints.get(serverless_like_endpoint_name)
serverless_like_scoring_uri = serverless_like_endpoint_details.scoring_uri

# Get the primary key
serverless_like_keys = ml_client.online_endpoints.get_keys(serverless_like_endpoint_name)
serverless_like_primary_key = serverless_like_keys.primary_key

print("✓ Serverless-like Endpoint credentials retrieved")
print(f"  Scoring URI: {serverless_like_scoring_uri}")
print(f"  Primary Key: <PRIMARY_KEY>  # Hidden for security")
print("\n⚠️  IMPORTANT: Store these values securely")
print("  - Do not print serverless_like_primary_key in production")
print("  - Use Azure Key Vault or Databricks Secrets for storage")

---

## 🅾️ SECTION 6 — Deploy Model to Batch Endpoint (C)

Batch Endpoints are designed for large-scale, asynchronous scoring jobs.
Ideal for processing large datasets where real-time inference is not required.

**Steps:**
1. Create a Batch Endpoint
2. Create a Batch Deployment
3. Invoke a batch scoring job
4. Check job status

### Step 6.1: Create Batch Endpoint

In [ ]:
from azure.ai.ml.entities import BatchEndpoint

# Define batch endpoint name - Replace with your desired name
batch_endpoint_name = "<BATCH_ENDPOINT_NAME>"  # e.g., "iris-batch-endpoint"

# Create Batch Endpoint
batch_endpoint = BatchEndpoint(
    name=batch_endpoint_name,
    description="Batch endpoint for Iris classifier",
    tags={"model": "iris-classifier", "type": "batch"}
)

print(f"Creating Batch Endpoint: {batch_endpoint_name}...")
batch_endpoint_result = ml_client.batch_endpoints.begin_create_or_update(batch_endpoint).result()

print("✓ Batch Endpoint created successfully")
print(f"  Name: {batch_endpoint_result.name}")
print(f"  Scoring URI: {batch_endpoint_result.scoring_uri}")

### Step 6.2: Create Batch Deployment

In [ ]:
from azure.ai.ml.entities import BatchDeployment, BatchRetrySettings
from azure.ai.ml.constants import BatchDeploymentOutputAction

# Create batch deployment
batch_deployment = BatchDeployment(
    name="blue",
    endpoint_name=batch_endpoint_name,
    model=registered_model.id,
    code_configuration=CodeConfiguration(
        code="model",
        scoring_script="score.py"
    ),
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu:1",
    compute="cpu-cluster",  # Replace with your compute cluster name
    instance_count=2,
    max_concurrency_per_instance=2,
    mini_batch_size=10,
    output_action=BatchDeploymentOutputAction.APPEND_ROW,
    output_file_name="predictions.csv",
    retry_settings=BatchRetrySettings(max_retries=3, timeout=300),
    logging_level="info",
)

print(f"Creating batch deployment 'blue' for endpoint {batch_endpoint_name}...")
print("This may take 10-15 minutes...")
batch_deployment_result = ml_client.batch_deployments.begin_create_or_update(batch_deployment).result()

print("✓ Batch Deployment created successfully")
print(f"  Deployment name: {batch_deployment_result.name}")
print(f"  Compute: {batch_deployment_result.compute}")
print(f"  Instance count: {batch_deployment_result.instance_count}")

### Step 6.3: Set Default Deployment

In [ ]:
# Set the blue deployment as default
batch_endpoint_result.defaults = {"deployment_name": "blue"}
ml_client.batch_endpoints.begin_create_or_update(batch_endpoint_result).result()

print("✓ Default deployment set to 'blue'")

### Step 6.4: Prepare Sample Batch Input Data

In [ ]:
import pandas as pd
import json

# Create sample batch input data
batch_data = pd.DataFrame({
    'sepal_length': [5.1, 6.2, 4.9, 5.8, 6.7],
    'sepal_width': [3.5, 3.4, 3.0, 2.7, 3.1],
    'petal_length': [1.4, 5.4, 1.4, 5.1, 5.6],
    'petal_width': [0.2, 2.3, 0.2, 1.9, 2.4]
})

# Save to local file
os.makedirs("batch_input", exist_ok=True)
batch_input_file = "batch_input/iris_batch.csv"
batch_data.to_csv(batch_input_file, index=False)

print("✓ Batch input data prepared")
print(f"  File: {batch_input_file}")
print(f"  Records: {len(batch_data)}")
print(f"\nSample data:")
print(batch_data.head())

### Step 6.5: Upload Batch Input to Datastore (Optional)

For production scenarios, upload your batch input data to an Azure ML datastore.

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Upload batch input data to datastore
batch_input_data = Data(
    path="batch_input",
    type=AssetTypes.URI_FOLDER,
    description="Batch input data for Iris classifier",
    name="iris-batch-input"
)

batch_input_data_asset = ml_client.data.create_or_update(batch_input_data)

print("✓ Batch input data uploaded to datastore")
print(f"  Data asset name: {batch_input_data_asset.name}")
print(f"  Version: {batch_input_data_asset.version}")
print(f"  Path: {batch_input_data_asset.path}")

### Step 6.6: Invoke Batch Scoring Job

In [ ]:
from azure.ai.ml import Input

# Invoke batch endpoint
job = ml_client.batch_endpoints.invoke(
    endpoint_name=batch_endpoint_name,
    deployment_name="blue",
    input=Input(path=batch_input_data_asset.id)
)

print("✓ Batch scoring job invoked")
print(f"  Job name: {job.name}")
print(f"  Job ID: {job.name}")
print(f"  Endpoint: {batch_endpoint_name}")
print(f"  Deployment: blue")
print(f"\n⏳ Job is running... Check status with the next cell")

### Step 6.7: Check Batch Job Status

In [ ]:
# Get job status
job_status = ml_client.jobs.get(job.name)

print("Batch Job Status:")
print(f"  Name: {job_status.name}")
print(f"  Status: {job_status.status}")
print(f"  Creation time: {job_status.creation_context.created_at}")

# Check if job is complete
if job_status.status == "Completed":
    print("\n✓ Job completed successfully!")
    print(f"  Output location: {job_status.outputs}")
elif job_status.status == "Failed":
    print("\n✗ Job failed")
    print(f"  Error: Check Azure ML Studio for details")
else:
    print(f"\n⏳ Job is still running... Current status: {job_status.status}")
    print("  Re-run this cell to check updated status")

### Step 6.8: Download Batch Job Results (Optional)

In [ ]:
# Download job outputs (run after job completes)
if job_status.status == "Completed":
    # Download outputs
    ml_client.jobs.download(name=job.name, download_path="./batch_output", output_name="score")
    
    print("✓ Batch job outputs downloaded")
    print("  Location: ./batch_output")
    
    # Display results if available
    try:
        results_df = pd.read_csv(f"./batch_output/named-outputs/score/predictions.csv")
        print(f"\nBatch Predictions ({len(results_df)} records):")
        print(results_df.head())
    except Exception as e:
        print(f"  Could not read results: {e}")
else:
    print("⏳ Job not yet completed. Wait for job to complete before downloading results.")

---

## 🧪 SECTION 7 — Consume Each Endpoint from Databricks

Now that all three endpoints are deployed, let's demonstrate how to consume each one from Databricks.

This section shows:
- **A.** Calling the Managed Online Endpoint
- **B.** Calling the Serverless-like Online Endpoint
- **C.** Triggering the Batch Endpoint

### 🅰️ A. Call Managed Online Endpoint

Send real-time scoring requests to the Managed Online Endpoint using REST API.

In [ ]:
import requests
import json

# Replace these with your actual values from Section 4.4
managed_endpoint_url = "<MANAGED_ENDPOINT_URL>"  # e.g., managed_scoring_uri from above
managed_api_key = "<PRIMARY_KEY>"  # e.g., managed_primary_key from above

# Prepare headers
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {managed_api_key}"
}

# Prepare test data (Iris features: sepal_length, sepal_width, petal_length, petal_width)
payload = {
    "input_data": [[5.1, 3.5, 1.4, 0.2]]  # Likely Iris-setosa
}

# Make POST request
print("Calling Managed Online Endpoint...")
response = requests.post(managed_endpoint_url, headers=headers, data=json.dumps(payload), timeout=30)

# Display results
print(f"\nStatus Code: {response.status_code}")
if response.status_code == 200:
    result = response.json()
    print("✓ Prediction successful!")
    print(f"  Response: {json.dumps(result, indent=2)}")
else:
    print(f"✗ Error: {response.text}")

### Test with Multiple Records

In [ ]:
# Test with multiple records
batch_payload = {
    "input_data": [
        [5.1, 3.5, 1.4, 0.2],  # Iris-setosa
        [6.2, 2.9, 4.3, 1.3],  # Iris-versicolor
        [7.3, 2.9, 6.3, 1.8],  # Iris-virginica
    ]
}

print("Testing with batch of 3 records...")
response = requests.post(managed_endpoint_url, headers=headers, data=json.dumps(batch_payload), timeout=30)

if response.status_code == 200:
    result = response.json()
    print("✓ Batch prediction successful!")
    print(f"  Predictions: {result.get('predictions', [])}")
    print(f"  Probabilities: {result.get('probabilities', [])}")
else:
    print(f"✗ Error: {response.text}")

---

### 🅱️ B. Call Serverless-like Online Endpoint

Send real-time scoring requests to the Serverless-like Online Endpoint.

In [ ]:
# Replace these with your actual values from Section 5.3
serverless_endpoint_url = "<SERVERLESS_ENDPOINT_URL>"  # e.g., serverless_like_scoring_uri from above
serverless_api_key = "<PRIMARY_KEY>"  # e.g., serverless_like_primary_key from above

# Prepare headers
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {serverless_api_key}"
}

# Prepare test data
payload = {
    "input_data": [[6.2, 2.9, 4.3, 1.3]]  # Likely Iris-versicolor
}

# Make POST request
print("Calling Serverless-like Online Endpoint...")
response = requests.post(serverless_endpoint_url, headers=headers, data=json.dumps(payload), timeout=30)

# Display results
print(f"\nStatus Code: {response.status_code}")
if response.status_code == 200:
    result = response.json()
    print("✓ Prediction successful!")
    print(f"  Response: {json.dumps(result, indent=2)}")
else:
    print(f"✗ Error: {response.text}")

---

### 🅾️ C. Trigger Batch Endpoint

Invoke the Batch Endpoint for large-scale scoring and monitor the job.

In [ ]:
# Invoke batch endpoint (using ML Client)
batch_endpoint_name = "<BATCH_ENDPOINT_NAME>"  # Replace with your batch endpoint name

print(f"Invoking Batch Endpoint: {batch_endpoint_name}...")

# Create new batch job
job = ml_client.batch_endpoints.invoke(
    endpoint_name=batch_endpoint_name,
    deployment_name="blue",
    input=Input(path="azureml://datastores/workspaceblobstore/paths/batch_input/")
)

print("✓ Batch job submitted successfully")
print(f"  Job name: {job.name}")
print(f"  Endpoint: {batch_endpoint_name}")
print(f"  Deployment: blue")

### Monitor Batch Job Status

In [ ]:
# Check batch job status
status = ml_client.jobs.get(job.name)

print("\nBatch Job Status:")
print(f"  Job Name: {status.name}")
print(f"  Status: {status.status}")
print(f"  Creation Time: {status.creation_context.created_at}")

# Status can be: NotStarted, Running, Completed, Failed, Canceled
if status.status == "Completed":
    print("\n✓ Batch job completed successfully!")
elif status.status == "Failed":
    print("\n✗ Batch job failed. Check Azure ML Studio for details.")
else:
    print(f"\n⏳ Job status: {status.status}")
    print("  Re-run this cell to check updated status")

### Alternative: Monitor with Loop (Optional)

In [ ]:
import time

# Monitor job until completion (with timeout)
print(f"Monitoring batch job: {job.name}\n")
timeout = 600  # 10 minutes
start_time = time.time()

while True:
    status = ml_client.jobs.get(job.name)
    print(f"[{time.strftime('%H:%M:%S')}] Status: {status.status}")
    
    if status.status in ["Completed", "Failed", "Canceled"]:
        break
    
    if time.time() - start_time > timeout:
        print("\n⏰ Timeout reached. Job is still running.")
        break
    
    time.sleep(30)  # Check every 30 seconds

print(f"\nFinal Status: {status.status}")
if status.status == "Completed":
    print("✓ Batch scoring completed successfully!")

---

## 📋 Summary

This notebook demonstrated:

### ✅ Completed Tasks

1. **Created and trained** a simple scikit-learn Iris classifier
2. **Deployed to Managed Online Endpoint (A)**
   - Real-time inference with managed infrastructure
   - Key-based authentication
   - REST API consumption

3. **Deployed to Serverless-like Endpoint (B)**
   - Alternative pattern for serverless behavior
   - Key-based authentication
   - REST API consumption

4. **Deployed to Batch Endpoint (C)**
   - Large-scale asynchronous scoring
   - Job-based processing
   - Output to datastore

5. **Demonstrated consumption** of all three endpoint types from Databricks

### 🔑 Key Takeaways

- **Managed Online Endpoints** are ideal for real-time inference with predictable traffic
- **Serverless Endpoints** (or serverless-like patterns) work well for unpredictable workloads
- **Batch Endpoints** are perfect for large-scale, non-real-time scoring
- All endpoints use **key-based authentication** for secure access
- Azure ML SDK v2 provides a **unified interface** for all deployment types

### 🔒 Security Best Practices

- ✅ Use **Azure Key Vault** or **Databricks Secrets** to store credentials
- ✅ Use **Managed Identity** for authentication when possible
- ✅ Rotate API keys regularly
- ✅ Use **environment variables** for configuration
- ✅ Never commit secrets to source control

### 🚀 Next Steps

- Monitor endpoint performance in Azure ML Studio
- Set up autoscaling for online endpoints
- Implement A/B testing with multiple deployments
- Add monitoring and alerting
- Integrate with MLOps pipelines

---

**📚 Resources:**
- [Azure ML SDK v2 Documentation](https://learn.microsoft.com/en-us/python/api/overview/azure/ai-ml-readme)
- [Online Endpoints](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-deploy-online-endpoints)
- [Batch Endpoints](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-use-batch-endpoint)
- [Azure ML Security](https://learn.microsoft.com/en-us/azure/machine-learning/concept-enterprise-security)